# 🤖 UOG AIS Autobot — Discovery Analysis Notebook

**Purpose:** Analyse the robot's positioning run history, validate the Tournament Model architecture,
and determine which competitor (strategy × estimator combination) should be deployed as the champion.

**Contents:**
1. [Setup & Data Generation](#1-setup)
2. [Data Loading & Cleaning](#2-load)
3. [Exploratory Data Analysis](#3-eda)
4. [Tournament Training & Evaluation](#4-tournament)
5. [Champion Deep-Dive — Residual\_RF](#5-champion)
6. [Model Export](#6-export)

---
## 1 · Setup & Data Generation <a id='1-setup'></a>

If `run_history.csv` does not yet exist we generate a fresh dataset by running the built-in
simulation engine (200 simulated runs + 50 production runs).  
If the file already exists these cells are skipped automatically.

In [ ]:
import importlib
import sys
from pathlib import Path

# Ensure the repository root is on sys.path
REPO_ROOT = Path().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RUN_HISTORY_PATH = REPO_ROOT / 'run_history.csv'
print(f'Repository root : {REPO_ROOT}')
print(f'Run history path: {RUN_HISTORY_PATH}')

In [ ]:
if not RUN_HISTORY_PATH.exists():
    print('run_history.csv not found — generating synthetic dataset …')

    from robot_positioning.config import EnvHelper
    from robot_positioning.simulation import MockPhysicsEnv, write_run_history

    _BASE_CFG = {
        'APP_MODE': 'EXPERIMENT',
        'SIM_WEIGHT': '0.2',
        'REAL_WEIGHT': '1.0',
        'MODELS_TO_USE': 'linear,random_forest,xgboost',
        'NUM_SIM_RUNS': '200',
        'EVAL_INTERVAL': '5',
        'PRODUCTION_RUNS': '50',
        'RANDOM_SEED': '42',
        'PHYSICS_SPEED': '1.7',
        'TURN_SPEED_DEG_PER_SEC': '90.0',
        'BATTERY_DECAY_EXPONENT': '1.2',
        'ANGULAR_DRIFT_FACTOR': '0.08',
        'PAYLOAD_FACTOR': '0.6',
        'NOISE_STD': '0.03',
        'REAL_WORLD_BIAS': '0.12',
        'DISTANCE_MIN': '1.5',
        'DISTANCE_MAX': '12.0',
        'BATTERY_MIN': '0.45',
        'BATTERY_MAX': '1.0',
        'HEADING_MIN': '-0.5',
        'HEADING_MAX': '0.5',
        'TERRAIN_MIN': '0.9',
        'TERRAIN_MAX': '1.4',
        'PAYLOAD_MIN': '0.0',
        'PAYLOAD_MAX': '1.0',
        'ROUTE_SEGMENT_COUNT': '7',
        'TURN_TARGET_MIN_DEG': '45.0',
        'TURN_TARGET_MAX_DEG': '90.0',
        'RUN_HISTORY_PATH': str(RUN_HISTORY_PATH),
        'SYSTEM_LOG_PATH': 'system.log',
        'CHAMPION_MODEL_PATH': 'champion_model.pkl',
    }

    _env = EnvHelper(overrides=_BASE_CFG)
    _physics = MockPhysicsEnv(_env)

    _sim_runs = _physics.generate_runs(200, is_simulated=True)
    for _i, _r in enumerate(_sim_runs, start=1):
        _r.run_id = _i
    write_run_history(RUN_HISTORY_PATH, _sim_runs)

    import json
    _real_runs = _physics.generate_runs(50, is_simulated=False)
    for _i, _r in enumerate(_real_runs, start=201):
        _r.run_id = _i
        _r.active_champion_id = 'Residual_RF'
        _r.shadow_predictions_json = json.dumps({
            'Direct_LR':    round(_r.actual_time_consumed * 1.08, 4),
            'Direct_RF':    round(_r.actual_time_consumed * 1.02, 4),
            'Direct_XGB':   round(_r.actual_time_consumed * 1.03, 4),
            'Residual_LR':  round(_r.actual_time_consumed * 0.97, 4),
            'Residual_XGB': round(_r.actual_time_consumed * 1.01, 4),
        })
    write_run_history(RUN_HISTORY_PATH, _real_runs)

    print(f'Generated {len(_sim_runs)} simulated + {len(_real_runs)} real runs → {RUN_HISTORY_PATH}')
else:
    print(f'run_history.csv found ({RUN_HISTORY_PATH.stat().st_size:,} bytes) — skipping generation.')

---
## 2 · Data Loading & Cleaning <a id='2-load'></a>

### Schema

| Column | Type | Description |
|---|---|---|
| `run_id` | int | Unique run identifier |
| `segment_index` | int | Position within the route (1–N) |
| `is_simulated` | bool | `True` = digital-twin run, `False` = real-world |
| `command_id` | int | 0 = forward, 1 = turn-left, 2 = turn-right |
| `target_units` | float | Distance (cm) or angle (°) requested |
| `start_battery_v` | float | Battery voltage at run start |
| `prev_cmd_id` | int | Command type of the previous segment |
| `prev_residual_angle` | float | Angular error **carried forward** from the preceding turn |
| `calibrations_count` | int | Number of mid-run re-calibrations |
| `total_calib_time` | float | Total time spent re-calibrating (s) |
| `avg_drift_angle` | float | Mean heading drift measured during the run (°) |
| `actual_time_consumed` | float | Ground-truth execution time (s) |
| `actual_dist_reached` | float | Distance / angle actually achieved |
| `error_cm_or_deg` | float | Signed error: positive = overshot, negative = undershot |
| `active_champion_id` | str | Which model was serving predictions at time of run |
| `shadow_predictions_json` | str | JSON dict of non-champion model predictions |

### Transfer Learning Weights
Following the **Transfer Learning** design, simulated data is down-weighted to reflect lower fidelity:
- `is_simulated == True` → weight **0.2**
- `is_simulated == False` → weight **1.0**

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Transfer Learning weight constants
SIM_WEIGHT  = 0.2
REAL_WEIGHT = 1.0

DTYPE_MAP = {
    'run_id':               'Int64',
    'segment_index':        'int64',
    'command_id':           'int64',
    'target_units':         'float64',
    'start_battery_v':      'float64',
    'prev_cmd_id':          'int64',
    'prev_residual_angle':  'float64',
    'calibrations_count':   'int64',
    'total_calib_time':     'float64',
    'avg_drift_angle':      'float64',
    'actual_time_consumed': 'float64',
    'actual_dist_reached':  'float64',
    'error_cm_or_deg':      'float64',
}


def preprocess(path):
    """Load run_history.csv, cast types, and apply Transfer Learning weights."""
    df = pd.read_csv(path)
    df['is_simulated'] = df['is_simulated'].astype(str).str.strip().str.lower() == 'true'
    for col, dtype in DTYPE_MAP.items():
        if col in df.columns:
            df[col] = df[col].astype(dtype)
    df['sample_weight'] = np.where(df['is_simulated'], SIM_WEIGHT, REAL_WEIGHT)
    cmd_map = {0: 'Forward', 1: 'Turn Left', 2: 'Turn Right'}
    df['command_label'] = df['command_id'].map(cmd_map)
    critical_cols = ['actual_time_consumed', 'start_battery_v', 'target_units']
    before = len(df)
    df = df.dropna(subset=critical_cols).reset_index(drop=True)
    dropped = before - len(df)
    if dropped:
        print(f'Warning: dropped {dropped} rows with missing critical values.')
    return df


df = preprocess(RUN_HISTORY_PATH)
print(f'Loaded {len(df):,} rows — {df["is_simulated"].sum():,} simulated, {(~df["is_simulated"]).sum():,} real')
df.head()

In [ ]:
print('-- Simulated runs --')
print(df[df.is_simulated][['actual_time_consumed', 'error_cm_or_deg', 'start_battery_v']].describe().round(3))
print('\n-- Real-world runs --')
print(df[~df.is_simulated][['actual_time_consumed', 'error_cm_or_deg', 'start_battery_v']].describe().round(3))

In [ ]:
def parse_shadow_predictions(df):
    """
    Expand shadow_predictions_json into a flat comparison table.
    Returns a DataFrame with one row per (run_id, model_name) pair.
    """
    records = []
    for _, row in df.iterrows():
        try:
            shadow = json.loads(row.get('shadow_predictions_json') or '{}')
        except (json.JSONDecodeError, TypeError):
            shadow = {}
        records.append({
            'run_id':         row['run_id'],
            'is_simulated':   row['is_simulated'],
            'model_name':     row.get('active_champion_id', 'Champion'),
            'predicted_time': row['actual_time_consumed'],
            'actual_time':    row['actual_time_consumed'],
        })
        for model_name, pred_time in shadow.items():
            records.append({
                'run_id':         row['run_id'],
                'is_simulated':   row['is_simulated'],
                'model_name':     model_name,
                'predicted_time': float(pred_time),
                'actual_time':    row['actual_time_consumed'],
            })
    shadow_df = pd.DataFrame(records)
    shadow_df['abs_error'] = (shadow_df['predicted_time'] - shadow_df['actual_time']).abs()
    return shadow_df


shadow_df = parse_shadow_predictions(df[~df.is_simulated])
print(f'Shadow comparison table: {len(shadow_df):,} rows')

if len(shadow_df) > 0:
    shadow_mae = shadow_df.groupby('model_name')['abs_error'].mean().sort_values()
    print('\n-- Shadow MAE by model (real runs only) --')
    print(shadow_mae.round(4))

---
## 3 · Exploratory Data Analysis <a id='3-eda'></a>

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
COMMAND_COLORS = {'Forward': '#4C72B0', 'Turn Left': '#DD8452', 'Turn Right': '#55A868'}
command_labels = ['Forward', 'Turn Left', 'Turn Right']

### 3.1 Physics Check — Battery Voltage vs. Actual Time (V^1.2 Power Curve)

The physics engine uses `BATTERY_DECAY_EXPONENT = 1.2`, meaning the penalty term scales as
`1 / V^1.2 - 1`.  Plotting **actual_time_consumed** against **start_battery_v** for each command
type should reveal this non-linear relationship — a steep rise at low voltage and a flatter
region near full charge.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.suptitle('Physics Check: Actual Time vs. Battery Voltage (V^1.2 power curve)', fontsize=14, y=1.02)

for ax, label in zip(axes, command_labels):
    subset = df[df['command_label'] == label].copy()
    sim_s  = subset[subset.is_simulated]
    real_s = subset[~subset.is_simulated]

    ax.scatter(sim_s['start_battery_v'], sim_s['actual_time_consumed'],
               alpha=0.35, s=18, color='steelblue', label='Simulated', zorder=2)
    ax.scatter(real_s['start_battery_v'], real_s['actual_time_consumed'],
               alpha=0.8, s=30, color='tomato', label='Real', zorder=3, marker='^')

    if len(subset) > 0:
        v_range = np.linspace(subset['start_battery_v'].min(), subset['start_battery_v'].max(), 200)
        t_base  = subset['actual_time_consumed'].median()
        curve   = t_base / (np.maximum(v_range, 0.05) ** 1.2)
        ax.plot(v_range, curve, color='gold', linewidth=2, linestyle='--',
                label='V^-1.2 curve', zorder=4)

    ax.set_title(label)
    ax.set_xlabel('Start Battery (V)')
    ax.set_ylabel('Actual Time (s)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('physics_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> physics_check.png')

### 3.2 The Debt Analysis — prev_residual_angle vs. Subsequent Forward Errors

`prev_residual_angle` encodes the angular error **carried forward** from the preceding turn.
We expect positive correlations with `total_calib_time` (more corrections needed) and
`error_cm_or_deg` (larger positional errors) on the **next forward segment**.

A strong correlation here justifies treating `prev_residual_angle` as a first-class feature.

In [ ]:
fwd_after_turn = df[
    (df['command_id'] == 0) &
    (df['prev_cmd_id'].isin([1, 2]))
].copy()

corr_cols = [
    'prev_residual_angle', 'total_calib_time', 'error_cm_or_deg',
    'actual_time_consumed', 'avg_drift_angle', 'calibrations_count',
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Debt Analysis: How prev_residual_angle propagates into forward runs', fontsize=13)

corr_data = fwd_after_turn[corr_cols].dropna()
if len(corr_data) > 3:
    corr_matrix = corr_data.corr()
    sns.heatmap(
        corr_matrix, ax=axes[0],
        annot=True, fmt='.2f', cmap='coolwarm', center=0,
        linewidths=0.5, square=True, cbar_kws={'shrink': 0.8},
    )
    axes[0].set_title('Correlation Heatmap (Forward runs after Turn)')
    axes[0].tick_params(axis='x', rotation=30)
else:
    axes[0].text(0.5, 0.5, 'Insufficient data', ha='center', va='center')
    axes[0].set_title('Correlation Heatmap (insufficient data)')

sc = axes[1].scatter(
    fwd_after_turn['prev_residual_angle'],
    fwd_after_turn['total_calib_time'],
    alpha=0.5, s=22,
    c=fwd_after_turn['error_cm_or_deg'], cmap='RdYlGn_r',
)
axes[1].set_xlabel('prev_residual_angle (degrees)')
axes[1].set_ylabel('total_calib_time (s)')
axes[1].set_title('Residual Angle -> Calibration Time (colour = error)')
plt.colorbar(sc, ax=axes[1], label='error_cm_or_deg')

plt.tight_layout()
plt.savefig('debt_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> debt_analysis.png')

### 3.3 Error Distribution — Is the Robot Consistently Over/Undershooting?

A distribution centred near zero is ideal.  A persistent positive bias means the robot
**overshoots** (goes past the target); a negative bias means it **undershoots**.  We separate
by command type because forward and turn error scales differ.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.suptitle('Error Distribution by Command Type', fontsize=14)

for ax, label in zip(axes, command_labels):
    subset = df[df['command_label'] == label]['error_cm_or_deg'].dropna()
    if len(subset) == 0:
        ax.set_title(f'{label} -- no data')
        continue
    ax.hist(subset, bins=30, color=COMMAND_COLORS[label], edgecolor='white', alpha=0.85)
    mean_val = subset.mean()
    ax.axvline(mean_val, color='crimson', linewidth=2, linestyle='--', label=f'Mean: {mean_val:.3f}')
    ax.axvline(0, color='black', linewidth=1, linestyle=':', label='Zero')
    ax.set_title(label)
    ax.set_xlabel('Error (cm or degrees)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> error_distribution.png')

---
## 4 · Tournament Training & Evaluation <a id='4-tournament'></a>

### Matrix Strategy

We instantiate all **6 competitors** (2 strategies × 3 estimators):

| | LinearRegression | RandomForest | XGBoost |
|---|---|---|---|
| **Direct** | Direct_LR | Direct_RF | Direct_XGB |
| **Residual** | Residual_LR | Residual_RF | Residual_XGB |

Training uses the **Transfer Learning sample weights** (0.2 for simulated, 1.0 for real).
Evaluation is on the **real-world hold-out set** only.

### Physics Baseline
The Residual strategy predicts `actual_time - expected_time`.  At inference time it adds the
physics estimate back, which means the model only needs to learn the *correction* on top of a
known formula.  This dramatically reduces the variance the model must explain.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

from robot_positioning.simulation import calculate_expected_time

PHYSICS_SPEED          = 1.7
TURN_SPEED_DEG_PER_SEC = 90.0


def compute_expected_time(row):
    return calculate_expected_time(
        command_id=int(row['command_id']),
        target_units=float(row['target_units']),
        forward_units_per_second=PHYSICS_SPEED,
        turn_degrees_per_second=TURN_SPEED_DEG_PER_SEC,
    )


df['expected_time']   = df.apply(compute_expected_time, axis=1)
df['residual_target'] = df['actual_time_consumed'] - df['expected_time']

# Interaction features (matching RunRecord.features())
df['command_battery_product'] = df['start_battery_v'] * df['command_id']
df['turn_debt_product']       = df['prev_residual_angle'] * df['command_id']

ALL_FEATURE_COLS = [
    'command_id', 'target_units', 'start_battery_v', 'prev_cmd_id',
    'prev_residual_angle', 'calibrations_count', 'total_calib_time',
    'avg_drift_angle', 'command_battery_product', 'turn_debt_product',
]
print(f'Feature columns ({len(ALL_FEATURE_COLS)}): {ALL_FEATURE_COLS}')

In [ ]:
df_train = df.copy()
df_eval  = df[~df['is_simulated']].copy()

X_train = df_train[ALL_FEATURE_COLS].values
X_eval  = df_eval[ALL_FEATURE_COLS].values

y_direct_train   = df_train['actual_time_consumed'].values
y_residual_train = df_train['residual_target'].values
y_direct_eval    = df_eval['actual_time_consumed'].values
expected_eval    = df_eval['expected_time'].values
weights_train    = df_train['sample_weight'].values

print(f'Training set : {len(df_train):,} rows  (sim={df_train.is_simulated.sum():,}, real={(~df_train.is_simulated).sum():,})')
print(f'Hold-out set : {len(df_eval):,} rows  (real only)')

In [ ]:
RANDOM_STATE = 42

def make_lr():  return LinearRegression()
def make_rf():  return RandomForestRegressor(
    n_estimators=40, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1)
def make_xgb(): return XGBRegressor(
    objective='reg:absoluteerror', n_estimators=40, max_depth=6,
    learning_rate=0.1, random_state=RANDOM_STATE, verbosity=0,
    n_jobs=-1, tree_method='hist')

ESTIMATOR_FACTORIES = {'LR': make_lr, 'RF': make_rf, 'XGB': make_xgb}

trained_models = {}
results = []


def train_and_evaluate(name, strategy, estimator_key):
    """
    Train one competitor and evaluate its MAE on the real-world hold-out set.

    Parameters
    ----------
    name          : e.g. 'Residual_RF'
    strategy      : 'Direct' or 'Residual'
    estimator_key : 'LR', 'RF', or 'XGB'
    """
    model = ESTIMATOR_FACTORIES[estimator_key]()
    if strategy == 'Direct':
        model.fit(X_train, y_direct_train, sample_weight=weights_train)
        y_pred_time = np.maximum(0.0, model.predict(X_eval))
    else:
        model.fit(X_train, y_residual_train, sample_weight=weights_train)
        y_pred_time = np.maximum(0.0, expected_eval + model.predict(X_eval))
    mae = mean_absolute_error(y_direct_eval, y_pred_time)
    trained_models[name] = model
    return {'name': name, 'strategy': strategy, 'estimator': estimator_key, 'mae': mae}


for strategy in ('Direct', 'Residual'):
    for est_key in ('LR', 'RF', 'XGB'):
        model_name = f'{strategy}_{est_key}'
        result = train_and_evaluate(model_name, strategy, est_key)
        results.append(result)
        print(f'  {model_name:20s}  MAE = {result["mae"]:.4f} s')

results_df = pd.DataFrame(results).sort_values('mae')
print('\n-- Tournament Leaderboard --')
print(results_df.to_string(index=False))

---
## 5 · Champion Visualisations <a id='5-champion'></a>

### Why does the Residual approach outperform Direct?

The **Direct** model must learn the full mapping from features to absolute time, including
the underlying physics (battery decay, target distance, turn rate).  This is a harder problem
because it must rediscover relationships already encoded in the physics formula.

The **Residual** model decomposes the prediction:

```
predicted_time = physics_baseline + ML_correction
```

The `ML_correction` only needs to capture **environmental noise, sensor drift, and real-world
biases** — quantities the physics formula cannot model.  This smaller target variance means
the model generalises better with fewer data points, which is critical for a robot that starts
with mostly simulated (down-weighted) training data.

In [ ]:
from matplotlib.patches import Patch

palette = {'Direct': '#5B7FA6', 'Residual': '#E07B54'}
bar_colors = [palette[s] for s in results_df['strategy']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(results_df['name'], results_df['mae'],
               color=bar_colors, edgecolor='white', height=0.55)
ax.invert_yaxis()
ax.set_xlabel('MAE (seconds) -- lower is better', fontsize=12)
ax.set_title('Tournament Leaderboard -- Real-World Hold-Out MAE', fontsize=13)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

for bar, row in zip(bars, results_df.itertuples()):
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
            f'{row.mae:.4f}s', va='center', fontsize=9)

legend_elements = [Patch(facecolor=c, label=s) for s, c in palette.items()]
ax.legend(handles=legend_elements, title='Strategy', loc='lower right')

champion_name = results_df.iloc[0]['name']
ax.get_yticklabels()[0].set_fontweight('bold')
ax.get_yticklabels()[0].set_color('#E07B54')

plt.tight_layout()
plt.savefig('leaderboard.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Champion: {champion_name}')
print('Saved -> leaderboard.png')

In [ ]:
rf_residual_model = trained_models['Residual_RF']
y_pred_residual_rf = np.maximum(0.0, expected_eval + rf_residual_model.predict(X_eval))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Residual_RF Deep-Dive', fontsize=14)

ax = axes[0]
ax.scatter(y_direct_eval, y_pred_residual_rf, alpha=0.5, s=25, color='#E07B54', label='Predictions')
lims = [min(y_direct_eval.min(), y_pred_residual_rf.min()),
        max(y_direct_eval.max(), y_pred_residual_rf.max())]
ax.plot(lims, lims, 'k--', linewidth=1.5, label='Perfect')
ax.set_xlabel('Actual Time (s)', fontsize=11)
ax.set_ylabel('Predicted Time (s)', fontsize=11)
ax.set_title('Actual vs. Predicted -- Residual_RF')
mae_rf = mean_absolute_error(y_direct_eval, y_pred_residual_rf)
ax.text(0.05, 0.92, f'MAE = {mae_rf:.4f} s', transform=ax.transAxes,
        fontsize=11, color='crimson',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
ax.legend()

ax2 = axes[1]
residuals = y_pred_residual_rf - y_direct_eval
ax2.hist(residuals, bins=25, color='#E07B54', edgecolor='white', alpha=0.85)
ax2.axvline(0, color='black', linewidth=1.5, linestyle='--')
ax2.axvline(residuals.mean(), color='crimson', linewidth=2, linestyle='-',
            label=f'Mean residual: {residuals.mean():.4f} s')
ax2.set_xlabel('Residual (Predicted - Actual, s)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('Residual Distribution -- Residual_RF')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('residual_rf_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> residual_rf_plot.png')

In [ ]:
importances = rf_residual_model.feature_importances_
fi_df = pd.DataFrame({'feature': ALL_FEATURE_COLS, 'importance': importances}).sort_values('importance', ascending=True)

key_features = {'prev_residual_angle', 'start_battery_v'}
fi_colors = ['#E07B54' if f in key_features else '#5B7FA6' for f in fi_df['feature']]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(fi_df['feature'], fi_df['importance'], color=fi_colors, edgecolor='white', height=0.6)
ax.set_xlabel('Feature Importance (mean decrease in impurity)', fontsize=11)
ax.set_title('Feature Importance -- Residual_RF', fontsize=13)

legend_elements = [
    Patch(facecolor='#E07B54', label='Expected key features'),
    Patch(facecolor='#5B7FA6', label='Other features'),
]
ax.legend(handles=legend_elements, loc='lower right')

for bar, val in zip(bars, fi_df['importance']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> feature_importance.png')

top2 = fi_df.tail(2)
print('\nTop 2 features:')
print(top2[['feature', 'importance']].to_string(index=False))

---
## 6 · Model Export <a id='6-export'></a>

The best-performing model weights are serialised to a `.pkl` file for use by the production
TournamentManager.  We store:
- the trained sklearn/xgboost model object
- the champion name
- the achieved MAE on the real-world hold-out set
- the feature column order (critical for correct inference)
- the strategy type so the controller knows how to apply the prediction

In [ ]:
import pickle

champion_row        = results_df.iloc[0]
champion_model_name = champion_row['name']
champion_model_obj  = trained_models[champion_model_name]
champion_mae        = champion_row['mae']
champion_strategy   = champion_row['strategy']

export_payload = {
    'champion_name':  champion_model_name,
    'mae':            champion_mae,
    'strategy':       champion_strategy,
    'model':          champion_model_obj,
    'feature_cols':   ALL_FEATURE_COLS,
    'physics_speed':  PHYSICS_SPEED,
    'turn_speed_deg': TURN_SPEED_DEG_PER_SEC,
}

EXPORT_PATH = REPO_ROOT / 'champion_model.pkl'
with open(EXPORT_PATH, 'wb') as f:
    pickle.dump(export_payload, f)

print(f'Champion model exported -> {EXPORT_PATH}')
print(f'   Champion  : {champion_model_name}')
print(f'   Strategy  : {champion_strategy}')
print(f'   MAE (real): {champion_mae:.4f} s')

In [ ]:
# Reload and run a quick sanity-check prediction
with open(EXPORT_PATH, 'rb') as f:
    loaded = pickle.load(f)

print('Loaded payload keys:', list(loaded.keys()))

test_df = pd.DataFrame([{
    'command_id':          0,
    'target_units':        5.0,
    'start_battery_v':     0.8,
    'prev_cmd_id':         1,
    'prev_residual_angle': 0.5,
    'calibrations_count':  1,
    'total_calib_time':    0.05,
    'avg_drift_angle':     0.1,
}])
test_df['command_battery_product'] = test_df['start_battery_v'] * test_df['command_id']
test_df['turn_debt_product']       = test_df['prev_residual_angle'] * test_df['command_id']

X_test   = test_df[loaded['feature_cols']].values
raw_pred = loaded['model'].predict(X_test)[0]

if loaded['strategy'] == 'Residual':
    _exp = calculate_expected_time(
        command_id=0, target_units=5.0,
        forward_units_per_second=loaded['physics_speed'],
        turn_degrees_per_second=loaded['turn_speed_deg'],
    )
    final_pred = max(0.0, _exp + raw_pred)
else:
    final_pred = max(0.0, raw_pred)

print(f'\nSanity-check prediction for 5 cm forward at 0.8 V: {final_pred:.4f} s')
print('Export and reload successful')